[Cell 1] 라이브러리 및 공통 평가 모듈 로드

In [ ]:
import os
import sys

# 현재 실행 위치(notebooks/)의 상위 폴더(Root)를 파이썬 탐색 경로에 추가
sys.path.append(os.path.dirname(os.path.abspath(os.getcwd())))

import re
import json
from typing import List, Dict, Any
# from tqdm.auto import tqdm
from tqdm.notebook import tqdm

# 문서 파싱 라이브러리
import fitz  # PyMuPDF (PDF용)
import docx  # python-docx (DOCX용)
import olefile  # 순수 파이썬 HWP 텍스트 추출용

# LangChain 핵심 구성 요소
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 공통 평가 모듈 로드
from src.evaluation.evaluator import RAGEvaluator
# 디바이스 유틸
from src.utils import get_device

# 평가 매니저 객체 초기화
evaluator = RAGEvaluator()
print("실전 베이스라인 파이프라인 및 평가 모듈 준비 완료!")

[Cell 2] STEP 1 & 2: 실제 다중 포맷 파일 로드 및 정규식 전처리 (베이스라인 청킹)

In [ ]:
def extract_text_from_pdf(file_path: str) -> str:
    """ PDF 파일에서 전체 텍스트를 추출합니다. """
    doc = fitz.open(file_path)
    text = ""
    for page in doc:
        text += page.get_text() + "\n"
    return text

def extract_text_from_docx(file_path: str) -> str:
    """ DOCX 파일에서 전체 텍스트를 추출합니다. """
    doc = docx.Document(file_path)
    return "\n".join([para.text for para in doc.paragraphs])

def extract_text_from_hwp(file_path: str) -> str:
    """ HWP 파일의 PrvText 스트림에서 텍스트를 추출합니다. (가장 가벼운 크로스플랫폼 방식) """
    try:
        f = olefile.OleFileIO(file_path)
        if f.exists('PrvText'):
            txt_data = f.openstream('PrvText').read()
            return txt_data.decode('utf-16le')
    except Exception as e:
        print(f"HWP 파싱 실패 ({file_path}): {e}")
    return ""

def load_and_preprocess_raw_files(data_folder_path: str) -> List[Document]:
    """ 
    실제 폴더 내의 hwp, docx, pdf 파일을 읽어와 
    3단계 정규식 전처리를 거친 뒤, 문서 단위로 기본 청킹을 수행하는 베이스라인 함수입니다.
    """
    documents = []
    
    if not os.path.exists(data_folder_path):
        print(f"경고: 데이터 폴더 '{data_folder_path}'가 존재하지 않습니다. 빈 리스트를 반환합니다.")
        return documents

    for file_name in os.listdir(data_folder_path):
        file_path = os.path.join(data_folder_path, file_name)
        ext = os.path.splitext(file_name)[-1].lower()
        
        raw_text = ""
        if ext == '.pdf':
            raw_text = extract_text_from_pdf(file_path)
        elif ext == '.docx':
            raw_text = extract_text_from_docx(file_path)
        elif ext == '.hwp':
            raw_text = extract_text_from_hwp(file_path)
        else:
            continue  # 지원하지 않는 확장자는 패스
            
        if not raw_text.strip():
            continue

        # ---------------------------------------------------------
        # [정규식 전처리 가이드라인]
        # ---------------------------------------------------------
        # 1) 불필요한 유니코드 및 비인쇄 제어 문자 제거 
        # (범위 오류를 막기 위해 아스키 제어문자 영역과 개별 유니코드 제어문자를 분리하여 제거합니다)
        cleaned_text = re.sub(r'[\x00-\x1f\x7f-\x9f]', '', raw_text) # 아스키 제어 문자 영역 제거
        cleaned_text = re.sub(r'[\u200b\u200c\u200d\uFEFF]', '', cleaned_text) # 제로폭 공백 등 특수 유니코드 제거
        
        # 2) 본문 분석에 방해되는 특수 기호 제거
        cleaned_text = re.sub(r'[■★◆●▲▶◀◆▼◇○□※]', '', cleaned_text)
        
        # 3) 과도한 공백 및 줄바꿈을 단일 공백으로 치환하여 문맥 연결성 확보
        cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()

        # 베이스라인 기술: 파일명을 기반으로 doc_id를 맵핑하여 텍스트 청크 생성 (문서 단위 청킹)
        doc_id = os.path.splitext(file_name)[0]  # 예: "20241001798.pdf" -> "20241001798"
        
        documents.append(Document(
            page_content=cleaned_text, 
            metadata={"doc_id": doc_id, "source": file_name}
        ))
        
    return documents

# 실제 RFP 원본 문서들이 모여있는 폴더 경로를 지정하세요!
raw_data_folder = '../data/raw/' 
print("Step 1 & 2: 실제 RFP 문서 로드 및 전처리 가동...")
processed_documents = load_and_preprocess_raw_files(raw_data_folder)
print(f" -> 완료: 총 {len(processed_documents)}개의 파일이 베이스라인 전처리되어 적재되었습니다.")

[Cell 3] STEP 3: nlpai-lab/KURE-v1 임베딩 및 FAISS 벡터 DB 적재

In [ ]:
print("Step 3: 지정된 공통 임베딩 모델(KURE-v1)을 활용하여 FAISS 벡터 DB 구축 중...")

# cpu, gpu, mps 해당 되는 디바이스 가져오기
device = get_device()

# MPS로 테스트 할 경우 메모리 할당 오류(Invalid buffer size)를 방지하기 위해 
# 임베딩 장치를 'cpu'로 안전하게 강제 고정 (참고자료이기에 사용)
# device = "cpu"

# 공통 임베딩 모델 설정
embedding_model = HuggingFaceEmbeddings(
    model_name="nlpai-lab/KURE-v1",
    model_kwargs={'device': device} # GPU 환경인 경우 'cuda'로 변경 가능
)

# 베이스라인 전처리된 문서가 비어있지 않은 경우 인덱스 빌드
if processed_documents:
    vector_db = FAISS.from_documents(processed_documents, embedding_model)
    # 자동 평가 지표(k=3) 요구사항에 맞춰 Retriever 생성
    retriever = vector_db.as_retriever(search_kwargs={"k": 3})
    print(" 완료: FAISS 인덱스 빌드 및 검색기(Retriever) 활성화 완료.")
else:
    print(" 적재할 문서가 없습니다. raw 폴더에 샘플 문서를 넣어주세요.")

[Cell 4] STEP 4: Qwen2-7B-Instruct 기반 RAG 체인 조립

In [ ]:
print("Step 4: 공통 LLM(Qwen2.5-1.5B-Instruct) 로드 및 RAG 파이프라인 조립...")

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, logging

# 하깅페이스의 모든 잔소리(경고 로그)를 강제로 꺼버리는 핵심 모듈
logging.set_verbosity_error()

from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# HuggingFace 원격 저장소의 공식 Qwen2 주소 설정 - (OSError: I/O error: IO Error: No space left on device)
# model_id = "Qwen/Qwen2-7B-Instruct"

# HuggingFace 원격 저장소의 공식 Qwen2 주소 설정 - 용량을 3GB 수준으로 우회용 고성능 경량 모델
model_id = "Qwen/Qwen2.5-1.5B-Instruct"

# 1. 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained(model_id, padding_side="left")

# 2. GPU 상에 모델 적재 (안전한 실전형 하이퍼파라미터 세팅)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float16,   # GPU 메모리를 2배 아끼기 위해 float16(반정밀도) 연산 적용
    device_map="auto"      # 다중 GPU 환경 혹은 단일 GPU 환경에 맞춰 메모리 레이아웃 자동 배치
)

# 3. 파이프라인과의 중복 인자 에러를 막기 위해 모델 내부에 심어진 기본 설정을 직접 수정합니다.
model.generation_config.max_new_tokens = 512
model.generation_config.max_length = None
model.generation_config.do_sample = False    # 일관된 팩트 체크 답변을 위해 샘플링 해제

# 최신 트랜스포머 스펙에 맞춰 do_sample=False와 모순되는 무작위성 옵션들을 모델 내부에서 제거
if hasattr(model.generation_config, "temperature"):
    delattr(model.generation_config, "temperature")
if hasattr(model.generation_config, "top_p"):
    delattr(model.generation_config, "top_p")

# 4. 트랜스포머 파이프라인 생성
hf_pipeline = pipeline(
    "text-generation", 
    model=model, 
    tokenizer=tokenizer
)

# 5. LangChain 표준 LCEL 규격에 완벽히 호환되는 핵심 LLM 객체 생성
# LangChain 내장 기본값(256)과의 충돌을 피하고 512 토큰 출력을 보장하도록 인자 맵핑
llm = HuggingFacePipeline(
    pipeline=hf_pipeline,
    pipeline_kwargs={"max_new_tokens": 512, "do_sample": False}
)

# 6. RFP 분석 전문가 프롬프트 정의
prompt_template = ChatPromptTemplate.from_template("""
당신은 제안요청서(RFP) 분석 전문가입니다. 주어진 참고 문서만을 바탕으로 질문에 정확하고 간결하게 답하십시오.
문서에 명시되지 않은 내용을 임의로 유추하거나 지어내어 답변하면 절대 안 됩니다.

[참고 문서]
{context}

[사용자 질문]
{question}

[전문가 답변]:""")

# 7. 컨텍스트 오버플로우 방지 및 안전 컷팅 함수
# 문서 3개가 합쳐져 모델 한계선인 131,072 토큰을 넘지 않도록 
# 검색된 각 문서의 앞부분 글자(25,000자)만 안전하게 잘라서 LLM에게 전달합니다.
def format_docs(docs):
    truncated_contents = []
    for d in docs:
        # 각 문서당 대략 1.5만 토큰 내외로 제한하여 3개 결합 시 오버플로우를 원천 차단합니다.
        truncated_contents.append(d.page_content[:25000])
    return "\n\n".join(truncated_contents)

# 8. 진짜 무기들이 장착된 최종 LangChain RAG 파이프라인 체인 조립
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt_template
    | llm
    | StrOutputParser()
)

print("완료: GPU 위에서 Qwen2 모델 적재 성공 및 LCEL RAG 체인 무결성 검증 완료!")

[Cell 5] STEP 5: 성능 평가 (모의고사 자동 채점 + 실제 질문 로그 적재)

In [ ]:
print("Step 5: 성능 평가 가동 (모의고사 자동 채점 + 수동 평가 양식 생성)\n")

import os
import json
from IPython.display import clear_output  # 서버 UI 먹통 현상을 치료할 화면 리로드 모듈

# 모의고사 답안지 파일 로드
eval_file_path = '../data/processed/eval/eval_dataset.json'

# 391문제를 다 돌리면 많은 시간이 소요됩니다. 
# 파이프라인 검증용으로 앞의 5개만 빠르게 테스트하려면 True, 전수 조사는 False로 바꾸세요.
test_mode = True 

if os.path.exists(eval_file_path) and processed_documents:
    print(f"'{eval_file_path}' 데이터를 기반으로 대량 자동 성능 평가를 시작합니다.")
    
    # 안전하게 파일 로드를 먼저 끝마치고 블록을 빠져나옵니다.
    with open(eval_file_path, 'r', encoding='utf-8') as f:
        full_dataset = json.load(f)
        
    # 테스트 모드 여부에 따른 데이터셋 슬라이싱 기본 제어
    if test_mode:
        real_dataset = full_dataset[:5]
        print(f"[테스트 모드 활성화] 전체 {len(full_dataset)}개 중 앞의 5개 문제만 샘플 채점합니다.")
    else:
        real_dataset = full_dataset
        print(f"[실전 전수 평가] 총 {len(full_dataset)}개의 문제를 모두 채점합니다. (시간이 소요됩니다)")
        
    eval_rows = []
    total_questions = len(real_dataset)
    
    # 커스텀 갱신 루프 tqdm 대신 주피터 자체 화면 리프레시 가동
    for idx, item in enumerate(real_dataset):
        current_num = idx + 1
        percent = (current_num / total_questions) * 100
        
        # 이전 화면 지우고 실시간 진행 상황을 깨끗하게 텍스트로 밀어내기
        clear_output(wait=True)
        print(f"[RAG 채점 파이프라인 가동률: {percent:.1f}%]")
        print(f"진행 상황: {current_num} / {total_questions} 문제 푸는 중 (NVIDIA L4 GPU 연산 중)")
        print(f"현재 평가 중인 질문: \"{item['question']}\"")
        print("-" * 70)
        
        # 실제 빌드된 RAG Retriever와 Chain을 통해 실시간 예측 데이터 추출
        retrieved_docs = retriever.invoke(item["question"])
        response_text = rag_chain.invoke(item["question"])
        
        row = item.copy()
        row["response"] = response_text
        # 수정된 evaluator 모듈 내부의 고유 ID 추출 메서드를 활용하여 완벽 매핑!
        row["retrieved_ids"] = [evaluator.get_doc_id(d) for d in retrieved_docs]
        eval_rows.append(row)
        
    # 채점 루프가 완벽히 끝나면 성적표 출력을 위해 화면을 마지막으로 한 번 더 정돈합니다.
    clear_output(wait=True)
    print("대량 자동 채점이 완료되었습니다! 베이스라인 최종 성적표를 산출합니다.\n")
        
    # 모듈화해둔 RAGEvaluator 기능을 호출하여 자동 정량 채점 수행 (k=3 전송)
    retrieval_scores = evaluator.evaluate_retrieval(eval_rows, k=3)
    gen_scores = evaluator.evaluate_generation(eval_rows)
    llm_judge_sample = evaluator.evaluate_llm_as_a_judge(eval_rows[0]["question"], eval_rows[0]["response"], eval_rows[0]["reference"])
    
    # 최종 성적표 출력
    print("=== [독립 평가] Retrieval (검색 성능 베이스라인 결과) ===")
    print(f" - HIT@3: {retrieval_scores['hits@3']:.4f}")
    print(f" - PRECISION@3: {retrieval_scores['precision@3']:.4f}")
    print(f" - RECALL@3: {retrieval_scores['recall@3']:.4f}")
    print(f" - MRR@3: {retrieval_scores['mrr@3']:.4f}")
    
    print("\n=== [독립 평가] Generator (생성 성능 베이스라인 결과) ===")
    print(f" - BLEU Score: {gen_scores['avg_bleu']:.4f}")
    print(f" - ROUGE-L Score: {gen_scores['avg_rougeL']:.4f}")
    
    print("\n=== [종단간 평가] End-to-End ===")
    print(f" - Token F1 Score: {gen_scores['avg_token_f1']:.4f}")
    print(f" - LLM-as-a-Judge Status: {llm_judge_sample}")

else:
    print("경고: 자동 평가를 진행하기 위한 eval_dataset.json 파일 또는 적재된 원본 문서가 부족합니다.")


# 5-2. 실제 사용자의 돌발 질문을 실시간 누적 기록하는 팀원 평가 로그 가동
if processed_documents:
    print("\n실제 사용자 돌발 질문에 대한 수동 채점 로그 적재 시뮬레이션...")
    user_surprise_question = "국민연금공단이 발주한 이러닝시스템 관련 사업 요구사항을 정리해 줘."

    live_retrieved = retriever.invoke(user_surprise_question)
    live_response = rag_chain.invoke(user_surprise_question)

    rag_result_for_log = {
        "response": live_response,
        "retrieved_contexts": [d.page_content for d in live_retrieved]
    }

    # 모듈화해둔 엑셀 추가 기능 호출
    evaluator.log_for_human_eval(user_surprise_question, rag_result_for_log, output_csv="real_user_eval_sheet.csv")
    print("\n실전 베이스라인 및 평가 파이프라인 검증 완료!")